# KOSIS BGE-M3 Top-K + Mapping-end 정식 비교 v2

READY 39건을 대상으로 Top-1·2·3·5를 비교합니다. 기존 BGE 검색 결과가 Drive에 있으면 자동 재사용하고, 수정된 단위·OBJ 검증부터 다시 실행합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/rnwjdgus03/NLP_05-Team-Project-3.git'
BRANCH = 'codex/gold-v1-lock'
REPO_DIR = Path('/content/NLP_05-Team-Project-3')
DRIVE_ROOT = Path('/content/drive/MyDrive/NLP_05-Team-Project-3')
INPUT_DIR = DRIVE_ROOT / 'inputs'
INDEX_DIR = DRIVE_ROOT / 'indexes' / 'kosis_bge_m3'
READY_CSV = INPUT_DIR / 'gold_measurement_scopefix_kosis_ready.csv'
GOLD_CSV = INPUT_DIR / 'gold_measurement_v1_locked.csv'
RUN_DIR = DRIVE_ROOT / 'runs' / 'bge_topk_mapping_end'

INPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import os
import sys

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-ml.txt'], check=True)
print('commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
from google.colab import files, userdata

required = [READY_CSV, GOLD_CSV]
missing = [path for path in required if not path.exists()]
if missing:
    print('선택할 파일:', [path.name for path in missing])
    uploaded = files.upload()
    for target in missing:
        if target.name in uploaded:
            target.write_bytes(uploaded[target.name])
remaining = [path for path in required if not path.exists()]
if remaining:
    raise FileNotFoundError(f'업로드되지 않은 파일: {[path.name for path in remaining]}')
manifest = INDEX_DIR / 'manifest.json'
if not manifest.exists():
    raise FileNotFoundError(f'BGE 인덱스가 없습니다: {manifest}')
if not os.environ.get('KOSIS_API_KEY'):
    os.environ['KOSIS_API_KEY'] = userdata.get('KOSIS_API_KEY')
if not os.environ.get('KOSIS_API_KEY'):
    raise RuntimeError('Colab 보안 비밀에 KOSIS_API_KEY를 등록하세요.')
print('입력·인덱스·API key 준비 완료')

In [ ]:
TABLE_INDEX = REPO_DIR / 'kosis_table_summary.csv'
stem = READY_CSV.stem
base_candidates = RUN_DIR / 'base_top5' / f'{stem}_kosis_candidates_with_meta.csv'
base_meta = RUN_DIR / 'base_top5' / f'{stem}_kosis_meta_index.csv'
reuse_base = base_candidates.exists() and base_meta.exists()
print('BGE 검색 결과 재사용:', reuse_base)

command = [
    sys.executable, str(REPO_DIR / 'run_kosis_topk_experiment.py'),
    '--input', str(READY_CSV),
    '--gold', str(GOLD_CSV),
    '--table-index', str(TABLE_INDEX),
    '--semantic-index', str(INDEX_DIR),
    '--out-dir', str(RUN_DIR),
    '--ks', '1', '2', '3', '5',
    '--semantic-top-k', '50',
    '--rerank-top-k', '20',
    '--device', 'cuda',
    '--delay', '0.05',
]
if reuse_base:
    command.append('--reuse-base')
subprocess.run(command, check=True)

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

summary = pd.read_csv(RUN_DIR / 'topk_summary.csv', encoding='utf-8-sig')
display(summary)
display(Markdown((RUN_DIR / 'topk_report.md').read_text(encoding='utf-8')))

technical = pd.read_csv(
    RUN_DIR / f'{READY_CSV.stem}_top5_technical_validated.csv',
    encoding='utf-8-sig',
)
print('\nmapping_status')
print(technical['mapping_status'].value_counts(dropna=False))
print('\nmapping_reason')
print(technical['mapping_reason'].value_counts(dropna=False).head(20))
print('\nAPI 진단')
for col in ['attempted_combination_count', 'api_valid_combination_count', 'api_error_count', 'empty_response_count']:
    print(col, pd.to_numeric(technical[col], errors='coerce').sum())